In [1]:
import pandas as pd

In [2]:
data=pd.read_csv("clean_resume_data.csv")

In [3]:
data.head()

,ID,Category,Feature
0,16852973,HR,hr administrator marketing associate hr admini...
1,22323967,HR,hr specialist hr operations summary media prof...
2,33176873,HR,hr director summary years experience recruitin...
3,27018550,HR,hr specialist summary dedicated driven dynamic...
4,17812897,HR,hr manager skill highlights hr skills hr depar...


In [4]:
data.columns

Index(['ID', 'Category', 'Feature'], dtype='object')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2684 entries, 0 to 2683
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID        2684 non-null   int64 
 1   Category  2684 non-null   object
 2   Feature   2683 non-null   object
dtypes: int64(1), object(2)
memory usage: 63.0+ KB


In [6]:
data.isnull().sum()

ID          0
Category    0
Feature     1
dtype: int64

In [7]:
data=data.dropna(subset=["Feature","Category"])

In [8]:
data=data.drop_duplicates()

In [9]:
data.shape

(2683, 3)

In [10]:
data["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    320
BUSINESS-DEVELOPMENT      119
FINANCE                   118
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
CHEF                      118
AVIATION                  117
FITNESS                   117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [11]:
import re

In [12]:
def cleanResume(text):
    text = str(text).lower()

    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\S+", " ", text)
    text = re.sub(r"#\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [13]:
data["Feature"]=data["Feature"].apply(cleanResume)

In [14]:
x=data["Feature"]
y=data["Category"]

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [18]:
tfidf=TfidfVectorizer(max_features=5000,stop_words="english",ngram_range=(1,2))

In [19]:
X_train_tfidf=tfidf.fit_transform(x_train)
X_test_tfidf=tfidf.transform(x_test)

In [20]:
from sklearn.ensemble import RandomForestClassifier

In [21]:
model=RandomForestClassifier(n_estimators=300,random_state=42,class_weight="balanced")

In [ ]:
model.fit(X_train_tfidf,y_train)

In [ ]:
y_pred=model.predict(X_test_tfidf)

In [24]:
from sklearn.metrics import accuracy_score

In [25]:
print("Accuracy:",accuracy_score(y_test,y_pred))

Accuracy: 0.7914338919925512


In [26]:
from sklearn.metrics import classification_report

In [27]:
print("Classification:",classification_report(y_test,y_pred))

Classification:                         precision    recall  f1-score   support

            ACCOUNTANT       0.76      0.95      0.84        20
              ADVOCATE       0.82      0.79      0.81        29
           AGRICULTURE       0.67      0.40      0.50        10
               APPAREL       0.80      0.53      0.64        15
                  ARTS       0.71      0.24      0.36        21
            AUTOMOBILE       1.00      0.33      0.50         3
              AVIATION       0.73      0.73      0.73        22
               BANKING       0.80      0.70      0.74        23
                   BPO       0.00      0.00      0.00         5
  BUSINESS-DEVELOPMENT       0.90      0.86      0.88        22
                  CHEF       0.84      0.96      0.90        27
          CONSTRUCTION       0.84      0.91      0.88        23
            CONSULTANT       0.88      0.70      0.78        20
              DESIGNER       0.95      0.91      0.93        23
         DIGITAL-MEDIA 

C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(averag

In [28]:
def predict_category(resume_text):
    resume_text=cleanResume(resume_text)
    resume_tfidf=tfidf.transform([resume_text])
    return model.predict(resume_tfidf)[0]

In [29]:
resume = """
Python
Pandas
NumPy
Machine Learning
Deep Learning
Scikit-learn
SQL
Jupyter Notebook
"""

print(predict_category(resume))

INFORMATION-TECHNOLOGY


In [30]:
import joblib

In [34]:
joblib.dump(model,"rf_classifier_categorization.joblib")
joblib.dump(tfidf,"tfidf_vectorizer_categorization.joblib")


['tfidf_vectorizer_categorization.joblib']